In [6]:
import cv2
import mediapipe as mp
import os
import csv
import time
import numpy as np

In [8]:
# Configurações do MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

# Configurações de classes
classes = [f"class_{i}" for i in range(1, 13)]  # Nomes das classes
output_dir = "dataset_pose"
os.makedirs(output_dir, exist_ok=True)

# Criação de pastas para cada classe
class_dirs = {}
for class_name in classes:
    class_path = os.path.join(output_dir, class_name)
    os.makedirs(class_path, exist_ok=True)
    class_dirs[class_name] = class_path

# Criação do arquivo CSV para salvar coordenadas
csv_file = os.path.join(output_dir, "pose_landmarks.csv")
with open(csv_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    # Cabeçalhos: ID da imagem, classe e coordenadas dos pontos de referência
    header = ["image_id", "class"]
    for i in range(33):  # 33 pontos para pose
        header.extend([f"x{i}", f"y{i}", f"z{i}", f"visibility{i}"])
    writer.writerow(header)

# Escolha da classe para capturar
print("Classes disponíveis:")
for idx, class_name in enumerate(classes):
    print(f"{idx + 1}: {class_name}")

try:
    class_index = int(input("Escolha a classe para capturar (1-12): ")) - 1
    if class_index < 0 or class_index >= len(classes):
        raise ValueError("Índice de classe inválido.")
    selected_class = classes[class_index]
    print(f"Capturando imagens para a classe: {selected_class}")
except ValueError as e:
    print(f"Erro: {e}")
    exit(1)

# Inicializa a captura de vídeo
cap = cv2.VideoCapture(0)

image_id = 0  # Contador de imagens para a classe selecionada

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Erro ao capturar o frame.")
            break

        # Converte para RGB (MediaPipe usa RGB)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb_frame)

        # Cria uma imagem preta para desenhar os pontos e as linhas
        black_mask = np.zeros_like(frame)

        # Desenha os pontos de referência e conexões na máscara preta
        if results.pose_landmarks:
            h, w, _ = frame.shape
            landmark_points = []

            # Converte landmarks normalizados para coordenadas em pixels
            for lm in results.pose_landmarks.landmark:
                x, y = int(lm.x * w), int(lm.y * h)
                landmark_points.append((x, y))
                cv2.circle(black_mask, (x, y), radius=5, color=(255, 255, 255), thickness=-1)  # Desenha os pontos

            # Desenha as conexões entre os pontos
            for connection in mp_pose.POSE_CONNECTIONS:
                start_idx, end_idx = connection
                if (
                    0 <= start_idx < len(landmark_points)
                    and 0 <= end_idx < len(landmark_points)
                ):
                    cv2.line(black_mask, landmark_points[start_idx], landmark_points[end_idx], (255, 255, 255), 2)

            # Coleta as coordenadas dos pontos
            landmarks = []
            for lm in results.pose_landmarks.landmark:
                landmarks.extend([lm.x * 1000, lm.y * 1000, lm.z * 1000, lm.visibility])  # Escala para facilitar leitura

            # Salva a imagem e os dados
            image_name = f"image_{image_id}.jpg"
            image_path = os.path.join(class_dirs[selected_class], image_name)
            if cv2.imwrite(image_path, black_mask):  # Verifica se a imagem foi salva
                # Salva os dados no CSV
                with open(csv_file, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([image_name, selected_class] + landmarks)
                print(f"Imagem {image_id} salva na classe '{selected_class}'.")
                image_id += 1
            else:
                print(f"Falha ao salvar a imagem {image_id}.")

        # Mostra a máscara preta com os pontos de referência e as conexões
        cv2.imshow(f"Pose Dataset - Classe: {selected_class}", black_mask)

        # Controle de taxa de captura
        time.sleep(0.1)  # Atraso de 100ms entre capturas

        # Sai com 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
except KeyboardInterrupt:
    print("Captura interrompida pelo usuário.")
finally:
    cap.release()
    cv2.destroyAllWindows()
    pose.close()

print(f"Dataset da classe '{selected_class}' salvo em: {class_dirs[selected_class]}")

Classes disponíveis:
1: class_1
2: class_2
3: class_3
4: class_4
5: class_5
6: class_6
7: class_7
8: class_8
9: class_9
10: class_10
11: class_11
12: class_12
Capturando imagens para a classe: class_1
Imagem 0 salva na classe 'class_1'.
Imagem 1 salva na classe 'class_1'.
Imagem 2 salva na classe 'class_1'.
Imagem 3 salva na classe 'class_1'.
Imagem 4 salva na classe 'class_1'.
Imagem 5 salva na classe 'class_1'.
Imagem 6 salva na classe 'class_1'.
Imagem 7 salva na classe 'class_1'.
Imagem 8 salva na classe 'class_1'.
Imagem 9 salva na classe 'class_1'.
Imagem 10 salva na classe 'class_1'.
Imagem 11 salva na classe 'class_1'.
Imagem 12 salva na classe 'class_1'.
Imagem 13 salva na classe 'class_1'.
Imagem 14 salva na classe 'class_1'.
Imagem 15 salva na classe 'class_1'.
Imagem 16 salva na classe 'class_1'.
Imagem 17 salva na classe 'class_1'.
Imagem 18 salva na classe 'class_1'.
Imagem 19 salva na classe 'class_1'.
Imagem 20 salva na classe 'class_1'.
Imagem 21 salva na classe 'class